In [1]:
import pandas as pd
import numpy as np

In [2]:
# Remove the ID column
df_raw = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

target_col = 'Status'

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]

In [3]:
from sklearn.preprocessing import LabelEncoder

y = LabelEncoder().fit_transform(y)

### Manually splitting the dataset

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f'Train lenght: {len(X_train)}')
print(f'Val lenght: {len(X_val)}')
print(f'Test lenght: {len(X_val)}')

Train lenght: 292
Val lenght: 63
Test lenght: 63


#### Pipelines

In [5]:
from sklearn.pipeline import Pipeline

In [6]:
num_cols = list(X.select_dtypes(include=['number']).columns)
cat_cols = list(X.select_dtypes(exclude=['number']).columns)

print(f'Num: {num_cols}')
print(f'Cat: {cat_cols}')

Num: ['Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage']
Cat: ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']


##### Num 

In [7]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline

num_si_mean_only_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean'))
])

num_knn_only_pipeline = Pipeline([
    ('knn imputer', KNNImputer())
])

num_ss_si_mean_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean')),
    ('standard scaler', StandardScaler())
])

num_mm_si_mean_pipeline = Pipeline([
    ('mean simple imputer', SimpleImputer(strategy='mean')),
    ('minmax scaler', MinMaxScaler())
])

num_ss_knn_pipeline = Pipeline([
    ('knn imputer', KNNImputer()),
    ('standard scaler', StandardScaler())
])

num_mm_knn_pipeline = Pipeline([
    ('knn imputer', KNNImputer()),
    ('minmax scaler', MinMaxScaler())
])

##### Cat

In [8]:
from sklearn.preprocessing import OneHotEncoder

cat_si_unspec_ohe_pipeline = Pipeline([
    ('Unpecified simple imputer', SimpleImputer(strategy='constant', fill_value='Unspecified')),
    ('one hot encoder', OneHotEncoder(handle_unknown='ignore'))
])

#### Training & Evaluation

In [9]:
num_pipelines = {
    'Mean Simple Imputer (No Scaling)': num_si_mean_only_pipeline,
    'Mean Simple Imputer + Standard Scaler': num_ss_si_mean_pipeline,
    'Mean Simple Imputer + MinMax Scaler': num_mm_si_mean_pipeline,
    'KNN Imputer (No Scaling)': num_knn_only_pipeline,
    'KNN Imputer + Standard Scaler': num_ss_knn_pipeline,
    'KNN Imputer + MinMax Scaler': num_mm_knn_pipeline
}

cat_pipelines = {
    'Constant Imputer (Unspecified) + OneHotEncoder': cat_si_unspec_ohe_pipeline
}

from prepare_models import create_default_models_dict

models = create_default_models_dict()

In [10]:
from utils import create_evaluation_dataframe

results_df = create_evaluation_dataframe(
    X_train,
    y_train,
    X_val,
    y_val,
    num_pipelines,
    cat_pipelines,
    models
)

results_df

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6690,1.0000,0.7302,1.0000,0.6936
1,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6690,1.0000,0.7302,1.0000,0.6936
2,KNN Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6690,1.0000,0.7302,1.0000,0.6936
3,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8048,0.7143,0.8162,0.6587,0.8048,0.7143,0.7822,0.6760
4,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8185,0.6984,0.8289,0.6442,0.8185,0.6984,0.7996,0.6580
5,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6825,1.0000,0.6313,1.0000,0.6825,1.0000,0.6452
6,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6667,1.0000,0.6145,1.0000,0.6667,1.0000,0.6319
7,Mean Simple Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6667,1.0000,0.6145,1.0000,0.6667,1.0000,0.6319
8,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7397,0.6508,0.6972,0.6022,0.7397,0.6508,0.7087,0.6071
9,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7363,0.6508,0.6942,0.6022,0.7363,0.6508,0.7049,0.6071


In [11]:
# We find rows without missing categorical data
mask_clean = X[cat_cols].notna().all(axis=1)

X_clean = X[mask_clean]
y_clean = y[mask_clean]

# New train/val/test split on cleaned data
X_train_cl_tmp, X_test_clean, y_train_cl_tmp, y_test_clean = train_test_split(X_clean, y_clean, test_size=0.15, random_state=42)
X_train_clean, X_val_clean, y_train_clean, y_val_clean = train_test_split(X_train_cl_tmp, y_train_cl_tmp, test_size=0.15/0.85, random_state=42)

print(f"X_train_clean: {X_train_clean.shape}")
print(f"X_val_clean: {X_val_clean.shape}")
print(f"X_test_clean: {X_test_clean.shape}")

X_train_clean: (218, 17)
X_val_clean: (47, 17)
X_test_clean: (47, 17)


In [12]:
# Pipeline for cleaned categorical data
from sklearn.preprocessing import OneHotEncoder

cat_pipelines_clean = {
    'Dropped missing Cat + OneHotEncoder': Pipeline([
        ('one hot encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
}

results_df_clean = create_evaluation_dataframe(
    X_train_clean,
    y_train_clean,
    X_val_clean,
    y_val_clean,
    num_pipelines,
    cat_pipelines_clean,
    models
)

results_df_clean

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1
0,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8349,0.7234,0.7890,0.6770,0.8349,0.7234,0.8106,0.6994
1,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8349,0.7234,0.7890,0.6770,0.8349,0.7234,0.8106,0.6994
2,KNN Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,GaussianNB,0.7431,0.7021,0.7852,0.7068,0.7431,0.7021,0.7426,0.6853
3,Mean Simple Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,GaussianNB,0.7431,0.7021,0.7852,0.7068,0.7431,0.7021,0.7426,0.6853
4,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7844,0.6809,0.7418,0.6339,0.7844,0.6809,0.7608,0.6532
5,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.6809,1.0000,0.6363,1.0000,0.6809,1.0000,0.6562
6,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.6809,1.0000,0.6363,1.0000,0.6809,1.0000,0.6562
7,Mean Simple Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,RandomForest,1.0000,0.6809,1.0000,0.6363,1.0000,0.6809,1.0000,0.6562
8,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7890,0.6809,0.7460,0.6339,0.7890,0.6809,0.7655,0.6532
9,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0000,0.6596,1.0000,0.6932,1.0000,0.6596,1.0000,0.6624


In [13]:
df_imputed = results_df.copy()
df_imputed['method'] = 'categorical_imputed'

df_dropped = results_df_clean.copy()
df_dropped['method'] = 'categorical_dropped'

combined_results = pd.concat([df_imputed, df_dropped], ignore_index=True)
display(combined_results)

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,method
0,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6690,1.0000,0.7302,1.0000,0.6936,categorical_imputed
1,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6690,1.0000,0.7302,1.0000,0.6936,categorical_imputed
2,KNN Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.7302,1.0000,0.6690,1.0000,0.7302,1.0000,0.6936,categorical_imputed
3,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8048,0.7143,0.8162,0.6587,0.8048,0.7143,0.7822,0.6760,categorical_imputed
4,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8185,0.6984,0.8289,0.6442,0.8185,0.6984,0.7996,0.6580,categorical_imputed
5,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6825,1.0000,0.6313,1.0000,0.6825,1.0000,0.6452,categorical_imputed
6,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6667,1.0000,0.6145,1.0000,0.6667,1.0000,0.6319,categorical_imputed
7,Mean Simple Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0000,0.6667,1.0000,0.6145,1.0000,0.6667,1.0000,0.6319,categorical_imputed
8,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7397,0.6508,0.6972,0.6022,0.7397,0.6508,0.7087,0.6071,categorical_imputed
9,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7363,0.6508,0.6942,0.6022,0.7363,0.6508,0.7049,0.6071,categorical_imputed


In [14]:
a = ['RandomForest', 'SVC', 'DecisionTree', 'GaussianNB']

for model in a:
    if model in combined_results['model'].values:
        display(combined_results[combined_results['model'] == model].sort_values('val_accuracy', ascending=False))
        print()

,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,method
0,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.7302,1.0,0.6690,1.0,0.7302,1.0,0.6936,categorical_imputed
1,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.7302,1.0,0.6690,1.0,0.7302,1.0,0.6936,categorical_imputed
2,KNN Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.7302,1.0,0.6690,1.0,0.7302,1.0,0.6936,categorical_imputed
5,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.6825,1.0,0.6313,1.0,0.6825,1.0,0.6452,categorical_imputed
30,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.6809,1.0,0.6363,1.0,0.6809,1.0,0.6562,categorical_dropped
29,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.6809,1.0,0.6363,1.0,0.6809,1.0,0.6562,categorical_dropped
31,Mean Simple Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.6809,1.0,0.6363,1.0,0.6809,1.0,0.6562,categorical_dropped
7,Mean Simple Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.6667,1.0,0.6145,1.0,0.6667,1.0,0.6319,categorical_imputed
6,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,RandomForest,1.0,0.6667,1.0,0.6145,1.0,0.6667,1.0,0.6319,categorical_imputed
36,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,RandomForest,1.0,0.6383,1.0,0.5955,1.0,0.6383,1.0,0.6147,categorical_dropped


,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,method
24,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8349,0.7234,0.7890,0.6770,0.8349,0.7234,0.8106,0.6994,categorical_dropped
25,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.8349,0.7234,0.7890,0.6770,0.8349,0.7234,0.8106,0.6994,categorical_dropped
3,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8048,0.7143,0.8162,0.6587,0.8048,0.7143,0.7822,0.6760,categorical_imputed
4,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.8185,0.6984,0.8289,0.6442,0.8185,0.6984,0.7996,0.6580,categorical_imputed
32,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7890,0.6809,0.7460,0.6339,0.7890,0.6809,0.7655,0.6532,categorical_dropped
28,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,SVC,0.7844,0.6809,0.7418,0.6339,0.7844,0.6809,0.7608,0.6532,categorical_dropped
9,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7363,0.6508,0.6942,0.6022,0.7363,0.6508,0.7049,0.6071,categorical_imputed
8,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.7397,0.6508,0.6972,0.6022,0.7397,0.6508,0.7087,0.6071,categorical_imputed
16,KNN Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.5822,0.5714,0.6057,0.6787,0.5822,0.5714,0.4526,0.4313,categorical_imputed
19,Mean Simple Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,SVC,0.5788,0.5556,0.5740,0.4922,0.5788,0.5556,0.4507,0.4227,categorical_imputed


,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,method
33,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6596,1.0,0.6932,1.0,0.6596,1.0,0.6624,categorical_dropped
35,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6383,1.0,0.6730,1.0,0.6383,1.0,0.6374,categorical_dropped
34,Mean Simple Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6383,1.0,0.6730,1.0,0.6383,1.0,0.6374,categorical_dropped
39,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6383,1.0,0.6496,1.0,0.6383,1.0,0.6259,categorical_dropped
40,KNN Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6170,1.0,0.6273,1.0,0.6170,1.0,0.6012,categorical_dropped
41,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,DecisionTree,1.0,0.6170,1.0,0.6273,1.0,0.6170,1.0,0.6012,categorical_dropped
12,KNN Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.5873,1.0,0.5646,1.0,0.5873,1.0,0.5752,categorical_imputed
11,KNN Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.5873,1.0,0.5646,1.0,0.5873,1.0,0.5752,categorical_imputed
13,KNN Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.5873,1.0,0.5646,1.0,0.5873,1.0,0.5752,categorical_imputed
15,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,DecisionTree,1.0,0.5714,1.0,0.5736,1.0,0.5714,1.0,0.5723,categorical_imputed


,num_pipeline,cat_pipeline,model,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,method
26,KNN Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,GaussianNB,0.7431,0.7021,0.7852,0.7068,0.7431,0.7021,0.7426,0.6853,categorical_dropped
27,Mean Simple Imputer (No Scaling),Dropped missing Cat + OneHotEncoder,GaussianNB,0.7431,0.7021,0.7852,0.7068,0.7431,0.7021,0.7426,0.6853,categorical_dropped
10,KNN Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.6986,0.6032,0.7251,0.6057,0.6986,0.6032,0.6966,0.5784,categorical_imputed
14,Mean Simple Imputer (No Scaling),Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.6986,0.5714,0.7391,0.6310,0.6986,0.5714,0.7041,0.5774,categorical_imputed
45,KNN Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2615,0.2340,0.7506,0.5493,0.2615,0.2340,0.3310,0.2862,categorical_dropped
44,Mean Simple Imputer + Standard Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2615,0.2340,0.7506,0.5493,0.2615,0.2340,0.3310,0.2862,categorical_dropped
46,Mean Simple Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2339,0.1915,0.7179,0.4870,0.2339,0.1915,0.2929,0.2332,categorical_dropped
47,KNN Imputer + MinMax Scaler,Dropped missing Cat + OneHotEncoder,GaussianNB,0.2339,0.1915,0.7179,0.4870,0.2339,0.1915,0.2929,0.2332,categorical_dropped
21,Mean Simple Imputer + MinMax Scaler,Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.1849,0.1905,0.6253,0.8354,0.1849,0.1905,0.1947,0.2123,categorical_imputed
20,Mean Simple Imputer + Standard Scaler,Constant Imputer (Unspecified) + OneHotEncoder,GaussianNB,0.1884,0.1905,0.5549,0.6590,0.1884,0.1905,0.1984,0.2267,categorical_imputed
